### This notebook is to create the database and compare the median income in the ACS Census data to the jurisdictions in the LMAS intake data.

The first thing I need to do is create the data tables that my database will be using by organizing the data into dataframes using pandas. I already completed an entity relationship diagram (ERD) that can be viewed in the Database folder, and will be following that to create the tables.

In [1]:
import pandas as pd
import sqlite3

In [2]:
lmas_df = pd.read_csv('../Data/animal_intake_data_2025.12.03.csv')
lmas_df.head()

,kennel,animalid,jurisdiction,intype,insubtype,indate,surreason,outtype,outsubtype,outdate,animaltype,sex,bites,petsize,color,breed,sourcezipcode,ObjectId
0,INTAKE,A430999,40204,OWNER SUR,EUTH REQ,2021-01-07 00:00:00,EUTH MED,EUTH,REQUESTED,2021-01-07 00:00:00,DOG,N,N,LARGE,WHITE / TAN,BASSET HOUND,40204,1
1,FOSTER,A423884,40211,STRAY,OTC,2025-03-07 00:00:00,STRAY,FOSTER,STRAY,2025-03-12 00:00:00,DOG,N,Y,SMALL,BLACK / TAN,MIN PINSCHER / MIN PINSCHER,40213,2
2,FOSTER,A423884,40213,FOSTER,RETURN,2025-04-13 00:00:00,FOSTER RET,ADOPTION,FOSTER,2025-04-13 00:00:00,DOG,N,Y,SMALL,BLACK / TAN,MIN PINSCHER / MIN PINSCHER,40213,3
3,N47,A583388,40228,OWNER SUR,OTC,2025-05-05 00:00:00,OWNER MED,ADOPTION,WALK IN,2025-05-20 00:00:00,CAT,N,N,X-LRG,ORANGE,DOMESTIC SH,40228,4
4,FREEZER,A558504,40215,STRAY,FIELD,2022-02-21 00:00:00,DOA,DISPOSAL,NECROPSY,2022-03-09 00:00:00,DOG,N,N,MED,BLUE,AMERICAN STAFF / MIX,40218,5


##### Animal Table

In [3]:
animal_df = lmas_df.drop(columns=['kennel', 'jurisdiction', 'intype', 'insubtype', 'indate', 'surreason', 'outtype', 'outsubtype', 'outdate', 'sourcezipcode', 'ObjectId'])
animal_df.head()

,animalid,animaltype,sex,bites,petsize,color,breed
0,A430999,DOG,N,N,LARGE,WHITE / TAN,BASSET HOUND
1,A423884,DOG,N,Y,SMALL,BLACK / TAN,MIN PINSCHER / MIN PINSCHER
2,A423884,DOG,N,Y,SMALL,BLACK / TAN,MIN PINSCHER / MIN PINSCHER
3,A583388,CAT,N,N,X-LRG,ORANGE,DOMESTIC SH
4,A558504,DOG,N,N,MED,BLUE,AMERICAN STAFF / MIX


Since animalid is duplicated per instance of intake, I am preemptively dropping the duplicate instances of the animal before loading it into a table specifically for animal, as I want to use animalid as the primary key. I will also update the column names.

In [4]:
animal_df.rename(columns={
    'animalid': 'animal_id',
    'animaltype': "animal_type",
    'petsize': 'size',
}, inplace=True)

animal_df.head()

,animal_id,animal_type,sex,bites,size,color,breed
0,A430999,DOG,N,N,LARGE,WHITE / TAN,BASSET HOUND
1,A423884,DOG,N,Y,SMALL,BLACK / TAN,MIN PINSCHER / MIN PINSCHER
2,A423884,DOG,N,Y,SMALL,BLACK / TAN,MIN PINSCHER / MIN PINSCHER
3,A583388,CAT,N,N,X-LRG,ORANGE,DOMESTIC SH
4,A558504,DOG,N,N,MED,BLUE,AMERICAN STAFF / MIX


In [5]:
animal_df.drop_duplicates(subset=None, keep='first', inplace=True, ignore_index=False)

animal_df.value_counts()

animal_id  animal_type  sex  bites  size   color              breed                      
A430999    DOG          N    N      LARGE  WHITE / TAN        BASSET HOUND                   1
A423884    DOG          N    Y      SMALL  BLACK / TAN        MIN PINSCHER / MIN PINSCHER    1
A583388    CAT          N    N      X-LRG  ORANGE             DOMESTIC SH                    1
A558504    DOG          N    N      MED    BLUE               AMERICAN STAFF / MIX           1
A477768    DOG          N    N      SMALL  BLACK / BLACK      CHIHUAHUA SH / MIX             1
                                                                                            ..
A774194    CAT          S    N      SMALL  BLACK              DOMESTIC MH                    1
A774196    CAT          N    N      LARGE  BRN TABBY / WHITE  DOMESTIC MH                    1
A774285    CAT          U    N      KITTN  BLACK              DOMESTIC SH                    1
A774389    CAT          U    N      MED    ORANGE      

In [6]:
animal_df[animal_df.duplicated()]

,animal_id,animal_type,sex,bites,size,color,breed


##### Intake Data

In [7]:
intake_df = lmas_df.drop(columns=['animaltype', 'sex', 'bites', 'petsize', 'color', 'breed', 'outtype', 'outsubtype', 'outdate'])

intake_df.head()

,kennel,animalid,jurisdiction,intype,insubtype,indate,surreason,sourcezipcode,ObjectId
0,INTAKE,A430999,40204,OWNER SUR,EUTH REQ,2021-01-07 00:00:00,EUTH MED,40204,1
1,FOSTER,A423884,40211,STRAY,OTC,2025-03-07 00:00:00,STRAY,40213,2
2,FOSTER,A423884,40213,FOSTER,RETURN,2025-04-13 00:00:00,FOSTER RET,40213,3
3,N47,A583388,40228,OWNER SUR,OTC,2025-05-05 00:00:00,OWNER MED,40228,4
4,FREEZER,A558504,40215,STRAY,FIELD,2022-02-21 00:00:00,DOA,40218,5


Unlike the lmas_intake_data notebook, I will be keeping the ObjectID as a primary key and renaming it to "occurence_id." I will also be renaming other columns. For this, I will not be dropping duplicate animal IDs since they can be returned or found again.

In [8]:
intake_df.rename(columns={
    'animalid': 'animal_id',
    'intype': 'intake_type',
    'insubtype': 'intake_subtype',
    'indate': "intake_date",
    'surreason': 'surrender_reason',
    'sourcezipcode': "source_zip_code",
    'ObjectId': 'occurence_id'
}, inplace=True)

intake_df.head()

,kennel,animal_id,jurisdiction,intake_type,intake_subtype,intake_date,surrender_reason,source_zip_code,occurence_id
0,INTAKE,A430999,40204,OWNER SUR,EUTH REQ,2021-01-07 00:00:00,EUTH MED,40204,1
1,FOSTER,A423884,40211,STRAY,OTC,2025-03-07 00:00:00,STRAY,40213,2
2,FOSTER,A423884,40213,FOSTER,RETURN,2025-04-13 00:00:00,FOSTER RET,40213,3
3,N47,A583388,40228,OWNER SUR,OTC,2025-05-05 00:00:00,OWNER MED,40228,4
4,FREEZER,A558504,40215,STRAY,FIELD,2022-02-21 00:00:00,DOA,40218,5


I will also be updating the type for the intake_date.

In [9]:
intake_df['intake_date'] = pd.to_datetime(intake_df['intake_date'])

intake_df.dtypes

kennel                         str
animal_id                      str
jurisdiction                   str
intake_type                    str
intake_subtype                 str
intake_date         datetime64[us]
surrender_reason               str
source_zip_code                str
occurence_id                 int64
dtype: object

##### Out Data

In [10]:
out_df = lmas_df.drop(columns=['kennel', 'animaltype', 'sex', 'bites', 'petsize', 'color', 'breed', 'jurisdiction', 'intype', 'insubtype', 'surreason', 'sourcezipcode', 'indate'])

out_df.head()

,animalid,outtype,outsubtype,outdate,ObjectId
0,A430999,EUTH,REQUESTED,2021-01-07 00:00:00,1
1,A423884,FOSTER,STRAY,2025-03-12 00:00:00,2
2,A423884,ADOPTION,FOSTER,2025-04-13 00:00:00,3
3,A583388,ADOPTION,WALK IN,2025-05-20 00:00:00,4
4,A558504,DISPOSAL,NECROPSY,2022-03-09 00:00:00,5


Like the Intake Data table, I am keeping ObjectId as occurence_id and using it as the primary key. I am also keeping animal_id and using it as a foreign key. I will also update the type of the outdate and the column names.

In [11]:
out_df.rename(columns={
    'animalid': 'animal_id',
    'outtype': 'out_type',
    'outsubtype': 'out_subtype',
    'outdate': 'out_date',
    'ObjectId': 'occurence_id'
}, inplace=True)

out_df.head()

,animal_id,out_type,out_subtype,out_date,occurence_id
0,A430999,EUTH,REQUESTED,2021-01-07 00:00:00,1
1,A423884,FOSTER,STRAY,2025-03-12 00:00:00,2
2,A423884,ADOPTION,FOSTER,2025-04-13 00:00:00,3
3,A583388,ADOPTION,WALK IN,2025-05-20 00:00:00,4
4,A558504,DISPOSAL,NECROPSY,2022-03-09 00:00:00,5


In [12]:
out_df['out_date'] = pd.to_datetime(out_df['out_date'])

out_df.dtypes

animal_id                  str
out_type                   str
out_subtype                str
out_date        datetime64[us]
occurence_id             int64
dtype: object

##### ACS Census Data

In [13]:
median_income_df = pd.read_csv('../Data/median_income_data.csv')

median_income_df.head()

,year,zip_code,median_income_val
0,2021-01-01,40203,26968
1,2021-01-01,40210,29733
2,2021-01-01,40059,161079
3,2021-01-01,40220,68598
4,2021-01-01,40047,82621


Please see the census_income_data notebook in the Notebooks folder to see the cleanup process of the median income data.

#### Creating the Database

In [14]:
import os

if os.path.exists("../Database/lmas_database.db"):
    os.remove("../Database/lmas_database.db")

conn = sqlite3.connect('../Database/lmas_database.db')

conn.execute("PRAGMA foreign_keys = ON;")

cursor = conn.cursor()

cursor.executescript("""
            CREATE TABLE IF NOT EXISTS animal (
                animal_id TEXT PRIMARY KEY,
                animal_type TEXT,
                breed TEXT,
                color TEXT,
                size TEXT,
                sex TEXT,
                bites TEXT
                );
            CREATE TABLE IF NOT EXISTS  intake_data (
                occurence_id INTEGER PRIMARY KEY,
                animal_id TEXT,
                intake_date TEXT,
                kennel TEXT,
                surrender_reason TEXT,
                intake_type TEXT,
                intake_subtype TEXT,
                jurisdiction TEXT,
                source_zip_code TEXT,
                FOREIGN KEY (animal_id) REFERENCES animal (animal_id)
                );
            CREATE TABLE IF NOT EXISTS out_data (
                occurence_id INTEGER PRIMARY KEY REFERENCES intake_data (occurence_id),
                animal_id TEXT,
                out_date TEXT,
                out_type TEXT,
                out_subtype TEXT,
                FOREIGN KEY (animal_id) REFERENCES animal (animal_id)   
                );
            CREATE TABLE IF NOT EXISTS median_income_data (
                zip_code TEXT PRIMARY KEY REFERENCES intake_data (jurisdiction),
                year TEXT,
                median_income_val INTEGER
                );
""")

In [15]:
animal_df.to_sql('animal', conn, if_exists='append', index=False)
intake_df.to_sql('intake_data', conn, if_exists='append', index=False)
out_df.to_sql('out_data', conn, if_exists='append', index=False)
median_income_df.to_sql('median_income', conn, if_exists='append', index=False)

conn.commit()


To check if the database successfully created, you can go to the Database folder in your explorer pane and look for "lmas_database.db" in the folder. If you have the extension "SQLite" by aleexcvzz installed, you can right click and open the database. To check if a table seeded correctly, run the next cell.

In [19]:
cursor.execute("SELECT COUNT(*) FROM animal;")
is_seeded = cursor.fetchone()[0] > 0

Now that the database is prepared, it's time to get started comparing the median income data to the lmas intake data.